In [1]:
import pandas as pd

X_train = pd.read_csv("Data/train_input_Z61KlZo.csv")
y_train = pd.read_csv("Data/train_output_DzPxaPY.csv")
X_test = pd.read_csv("Data/test_input_5qJzHrr.csv")
y_pred_random = pd.read_csv("Data/test_output_random.csv")

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

C:\Users\ilyes\AppData\Local\Temp\ipykernel_4980\461888069.py:3: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  X_train = pd.read_csv("Data/train_input_Z61KlZo.csv")
C:\Users\ilyes\AppData\Local\Temp\ipykernel_4980\461888069.py:5: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  X_test = pd.read_csv("Data/test_input_5qJzHrr.csv")


Train: (383610, 374)
Test: (95852, 374)


# Random

In [ ]:
# import numpy as np

# # local
# y_train_pred_random = y_train_test.copy()
# y_train_pred_random['CHARGE'] = np.random.choice(y_train_train['CHARGE'], size=len(y_train_test))

# rmse(y_train_test, y_train_pred_random)

# Zero

In [ ]:
y_pred_zero = y_pred_random.copy()
y_pred_zero['CHARGE'] = 0

y_pred_zero.to_csv('Submissions/submission_zero.csv', index=False)

# Preprocessing

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

def preprocessing(X_train, X_test):
    # Save ID columns
    train_ids = X_train['ID']
    test_ids = X_test['ID']

    # separate columns + remove 'ID'
    num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    num_cols.remove('ID')
    cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    # reduce size
    X_train[num_cols] = X_train[num_cols].astype("float32")
    X_test[num_cols] = X_test[num_cols].astype("float32")

    # mixed types -> strings only
    X_train[cat_cols] = X_train[cat_cols].astype(str)
    X_test[cat_cols] = X_test[cat_cols].astype(str)

    # pipelines
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

    # fit transform
    X_train_clean = preprocessor.fit_transform(X_train)
    X_test_clean = preprocessor.transform(X_test)

    # Feature names
    all_features = num_cols + cat_cols

    # dataframes
    X_train_clean = pd.DataFrame(X_train_clean, columns=all_features, index=train_ids)
    X_test_clean = pd.DataFrame(X_test_clean, columns=all_features, index=test_ids)

    return X_train_clean, X_test_clean

In [4]:
# X_train_clean, X_test_clean = preprocessing(X_train, X_test)

# X_train_clean.to_csv('Data/X_train_clean.csv')
# X_test_clean.to_csv('Data/X_test_clean.csv')

import pandas as pd

X_train_clean = pd.read_csv('Data/X_train_clean.csv', index_col='ID')
X_test_clean = pd.read_csv('Data/X_test_clean.csv', index_col='ID')

# LASSO

In [6]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

y_pred_lasso = y_pred_random.copy()
target = 'CHARGE'
y_train_col = y_train[target]
model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=50, random_state=42) # alpha up to 100
)
model.fit(X_train_clean, y_train_col)
preds = model.predict(X_test_clean)
y_pred_lasso[target] = preds

y_pred_lasso.to_csv('Submissions/submission_lasso50.csv', index=False)

MemoryError: Unable to allocate 1.07 GiB for an array with shape (383610, 373) and data type float64

# Decision Trees

In [ ]:
from sklearn.tree import DecisionTreeRegressor

y_pred_dts = y_pred_random.copy()

targets = ['FREQ', 'CM', 'ANNEE_ASSURANCE']

for target in targets:
    y_train_col = y_train[target]
    model = DecisionTreeRegressor(max_depth=5, random_state=42)
    model.fit(X_train_clean, y_train_col)
    preds = model.predict(X_test_clean)
    y_pred_dts[target] = preds

y_pred_dts['CHARGE'] = y_pred_dts['FREQ'] * y_pred_dts['CM'] * y_pred_dts['ANNEE_ASSURANCE']

y_pred_dts.to_csv('Submissions/submission_dts5.csv', index=False)

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

y_pred_dt = y_pred_random.copy()

target = 'CHARGE'

y_train_col = y_train[target]
model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(X_train_clean, y_train_col)
preds = model.predict(X_test_clean)
y_pred_dt[target] = preds

y_pred_dt.to_csv('Submissions/submission_dt5.csv', index=False)

# Hybrid Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

def hdt(X_train_train_clean, X_train_test_clean, y_train_train, y_train_test):
    y_train_train = y_train_train.set_index('ID').copy()
    y_train_test = y_train_test.set_index('ID').copy()

    target = 'CHARGE'

    # Step 1: Binary version of the target
    y_train_target = y_train_train[[target]]
    y_train_binary = (y_train_target != 0).astype(int)

    # Step 2: Train Decision Tree Classifier
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    clf.fit(X_train_train_clean, y_train_binary)

    # Step 3: Predict binary labels for test set
    binary_preds = clf.predict(X_train_test_clean)

    # Step 4: Train Regressor on data where target != 0
    non_zero_indices = y_train_binary[y_train_binary == 1].index
    reg = DecisionTreeRegressor(max_depth=5, random_state=42)
    reg.fit(X_train_train_clean.loc[non_zero_indices], y_train_target.loc[non_zero_indices])

    # Step 5: Initialize predictions
    y_train_pred_hdt = y_train_test.copy()
    y_train_pred_hdt[target] = 0  # default prediction is 0

    # Step 6: Apply regression only where classifier predicted 1
    test_non_zero_indices = X_train_test_clean.index[binary_preds == 1]
    y_train_pred_hdt.loc[test_non_zero_indices, target] = reg.predict(X_train_test_clean.loc[test_non_zero_indices])

    return y_train_pred_hdt.reset_index()

In [ ]:
y_pred_hdt = hdt(X_train_clean, X_test_clean, y_train, y_pred_random)
y_pred_hdt.to_csv('Submissions/submission_hdt5.csv', index=False)

# Decision Tree & Random Forest

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor  # updated import

def dtrf(X_train_train_clean, X_train_test_clean, y_train_train, y_train_test):
    y_train_train = y_train_train.set_index('ID').copy()
    y_train_test = y_train_test.set_index('ID').copy()

    target = 'CHARGE'

    # Step 1: Binary version of the target
    y_train_target = y_train_train[[target]]
    y_train_binary = (y_train_target != 0).astype(int)

    # Step 2: Train Decision Tree Classifier
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    clf.fit(X_train_train_clean, y_train_binary)

    # Step 3: Predict binary labels for test set
    binary_preds = clf.predict(X_train_test_clean)

    # Step 4: Train Random Forest Regressor on data where target != 0
    non_zero_indices = y_train_binary[y_train_binary == 1].index
    reg = RandomForestRegressor(max_depth=5, random_state=42)
    reg.fit(X_train_train_clean.loc[non_zero_indices], y_train_target.loc[non_zero_indices].values.ravel())

    # Step 5: Initialize predictions
    y_train_pred_dtrf = y_train_test.copy()
    y_train_pred_dtrf[target] = 0  # default prediction is 0

    # Step 6: Apply regression only where classifier predicted 1
    test_non_zero_indices = X_train_test_clean.index[binary_preds == 1]
    y_train_pred_dtrf.loc[test_non_zero_indices, target] = reg.predict(X_train_test_clean.loc[test_non_zero_indices])

    return y_train_pred_dtrf.reset_index()

In [ ]:
y_pred_dtrf = dtrf(X_train_clean, X_test_clean, y_train, y_pred_random)
y_pred_dtrf.to_csv('Submissions/submission_dt5rf5.csv', index=False)